## [1581. Customer Who Visited but Did Not Make Any Transactions](https://leetcode.com/problems/customer-who-visited-but-did-not-make-any-transactions/description/?envType=study-plan-v2&envId=top-sql-50)

In [15]:
import os

!pip install -q pyspark==3.4.1 #delta-spark==2.4.0

In [16]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()

### Example 1

#### Input

**Visits**

| visit_id | customer_id |
|----------|-------------|
| 1 | 23 |
| 2 | 9 |
| 4 | 30 |
| 5 | 54 |
| 6 | 96 |
| 7 | 54 |
| 8 | 54 |

**Transactions**

| transaction_id | visit_id | amount |
|----------------|----------|--------|
| 2 | 5 | 310 |
| 3 | 5 | 300 |
| 9 | 5 | 200 |
| 12 | 1 | 910 |
| 13 | 2 | 970 |

#### Output

| customer_id | count_no_trans |
|-------------|----------------|
| 54 | 2 |
| 30 | 1 |
| 96 | 1 |

#### Explanation

- Customer `23` visited once and made one transaction.
- Customer `9` visited once and made one transaction.
- Customer `30` visited once and made no transactions.
- Customer `54` visited three times. Two visits had no transactions, and one visit had three transactions.
- Customer `96` visited once and made no transactions.

In [20]:
visits_data = [
    (1, 23),
    (2, 9),
    (4, 30),
    (5, 54),
    (6, 96),
    (7, 54),
    (8, 54)
]

visits_df = spark.createDataFrame(
    visits_data,
    ["visit_id", "customer_id"]
)

transactions_data = [
    (2, 5, 310),
    (3, 5, 300),
    (9, 5, 200),
    (12, 1, 910),
    (13, 2, 970)
]

transactions_df = spark.createDataFrame(
    transactions_data,
    ["transaction_id", "visit_id", "amount"]
)

visits_df.show()

transactions_df.show()

+--------+-----------+
|visit_id|customer_id|
+--------+-----------+
|       1|         23|
|       2|          9|
|       4|         30|
|       5|         54|
|       6|         96|
|       7|         54|
|       8|         54|
+--------+-----------+

+--------------+--------+------+
|transaction_id|visit_id|amount|
+--------------+--------+------+
|             2|       5|   310|
|             3|       5|   300|
|             9|       5|   200|
|            12|       1|   910|
|            13|       2|   970|
+--------------+--------+------+



In [22]:
visits_df.alias("vi").join(
    transactions_df.alias("tr"),
    (col("vi.visit_id")==col("tr.visit_id")),
    "left"
    ).filter(col("tr.transaction_id").isNull()
    ).groupBy(col("customer_id")).count().alias("count_no_trans").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|         54|    2|
|         96|    1|
|         30|    1|
+-----------+-----+

